# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
This notebook provides a guided exploration of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices for referencing entities via their `@id` fields.

### Dataset Source
The dataset is defined via a Croissant schema at the following URL:

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
We begin by loading both the metadata and the tabular records of the dataset using mlcroissant.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
ds = mlc.Dataset(croissant_url)

# Access the dataset metadata
meta = ds.metadata
print(f"Dataset Name: {getattr(meta, 'name', 'N/A')}")
print(f"Description: {getattr(meta, 'description', 'N/A')}")

## 2. Data Overview
Let's review which record sets (`cr:RecordSet`) are present, along with their fields and available columns.
We'll reference each by its `@id`.

In [ ]:
# Retrieve all record sets and print a summary using `@id`
record_sets = getattr(meta, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for record_set in record_sets:
        print(f"Record Set: {getattr(record_set, '@id', 'N/A')}")
        fields = getattr(record_set, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            print(f"  Field: {getattr(field, '@id', 'N/A')}")
            columns = getattr(field, 'column', [])
            if not isinstance(columns, list):
                columns = [columns]
            for column in columns:
                print(f"    Column: {getattr(column, '@id', 'N/A')}")

## 3. Data Extraction
We now load one or more record sets into DataFrames for analysis. Use the `@id` values printed above to specify the record set(s) of interest. For this notebook, we'll auto-select the first available record set as an example.

In [ ]:
# Extract all record set @id's
record_sets = [getattr(rs, '@id') for rs in getattr(meta, 'recordSet', [])]
if not record_sets:
    raise ValueError("No record sets found in the dataset metadata.")

# Load all (or selected) record sets as DataFrames, using @id explicitly
dataframes = {}
for record_set_id in record_sets:
    # List of dicts for records
    records = list(ds.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} with shape {dataframes[record_set_id].shape}")
    else:
        print(f"No records found for {record_set_id}.")

# For demonstration, select the first record set
main_record_set_id = record_sets[0]
if main_record_set_id in dataframes:
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame loaded for {main_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the main record set. We'll demonstrate filtering, normalization, and grouped aggregations using field `@id`s.

In [ ]:
# We'll attempt to select a numeric field and a potential grouping field
# Use field @id's for column access. We'll pick the first float/integer-typed column if available.
main_df = dataframes[main_record_set_id]
numeric_field = None
group_field = None

# Attempt to infer types via metadata if available, or else check pandas dtypes
fields = []
for rs in getattr(meta, 'recordSet', []):
    if getattr(rs, '@id') == main_record_set_id:
        fields = getattr(rs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        break

# Try to pick a numeric field by type or column dtype
for field in fields:
    field_id = getattr(field, '@id', None)
    # Try to find a numeric (float/int) type in Croissant
    data_type = getattr(field, 'dataType', '').lower() if hasattr(field, 'dataType') else ''
    if field_id and field_id in main_df.columns and (
        'integer' in data_type or 'float' in data_type or pd.api.types.is_numeric_dtype(main_df[field_id])
    ):
        numeric_field = field_id
        break
if not numeric_field:
    # Fallback: find the first numeric dtype column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break

# For grouping, pick the first non-numeric/top-level column with few unique values
for col in main_df.columns:
    if col != numeric_field and main_df[col].nunique() < max(10, len(main_df) // 4):
        group_field = col
        break

if numeric_field is None:
    print("No suitable numeric field found for EDA.")
else:
    threshold = main_df[numeric_field].quantile(0.75)
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile): {len(filtered_df)} records")

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Perform groupby if a suitable field exists
    if group_field and group_field in filtered_df.columns:
        group_stat = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        print(group_stat.head())
    else:
        print("No suitable group field for grouping."
              " Consider changing group_field above to a valid field @id.")

## 5. Visualization
We can plot the distribution of the selected numeric field and its grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field is not None and group_field in main_df.columns:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and process a FAIR dataset described by a Croissant schema via mlcroissant.

- All dataset, record set, field, and column references utilize explicit `@id` fields for reproducibility.
- We displayed both metadata and tabular records, performed simple filtering and normalization, and visualized distributions.

You can extend this workflow by exploring more fields, joining multiple record sets by `@id` keys, or running additional visualizations and machine learning models.